In [22]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = ['PLY.AX', 'LAU.AX', 'COS.AX', 'ANG.AX', 'VVA.AX',
           'WTC.AX', 'AUB.AX', 'TLX.AX', 'DUG.AX']

data = yf.download(tickers, period='5y', interval='1d')['Close']

daily_log_returns = np.log(data / data.shift(1))

monthly_prices = data.resample('ME').last()
monthly_returns = np.log(monthly_prices / monthly_prices.shift(1))[1:]

monthly_volatility = daily_log_returns.resample('ME').std()[1:]

result = pd.concat({
    'monthly_return': monthly_returns,
    'monthly_vol': monthly_volatility
}, axis=1).dropna()

print(monthly_returns.shape, monthly_volatility.shape)

[*********************100%***********************]  9 of 9 completed


(60, 9) (60, 9)


In [23]:
import statsmodels.api as sm

results = {}

for stock in monthly_returns.columns:
    y = monthly_returns[stock]
    x = monthly_volatility[stock]
    
    x = sm.add_constant(x)
    
    model = sm.OLS(y, x).fit()
    results[stock] = model
    
    print(f"\n{stock}")
    print(model.summary())


ANG.AX
                            OLS Regression Results                            
Dep. Variable:                 ANG.AX   R-squared:                       0.051
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     3.113
Date:                Sun, 19 Apr 2026   Prob (F-statistic):             0.0829
Time:                        19:45:15   Log-Likelihood:                 36.069
No. Observations:                  60   AIC:                            -68.14
Df Residuals:                      58   BIC:                            -63.95
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0837      0.047      1.784 

In [24]:
momentum = monthly_returns.rolling(window=3).sum().shift(1)
momentum = momentum.dropna()

volume_data = yf.download(tickers, period='5y', interval='1d')['Volume']
liquidity = np.log(volume_data.resample('ME').sum())

market_data = yf.download('^AXJO', period='5y', interval='1d')['Close']
market_data = market_data.resample('ME').last()
market_ret = np.log(market_data/market_data.shift(1)).dropna()

market_ret_df = pd.DataFrame(
    np.tile(market_ret.values.reshape(-1, 1), (1, len(tickers))),
    index=market_ret.index,
    columns=tickers
)

df = pd.concat({
    'return': monthly_returns,
    'vol': monthly_volatility,
    'momentum': momentum,
    'liquidity': liquidity,
    'market performance': market_ret_df
}, axis=1).dropna()


[*********************100%***********************]  9 of 9 completed
[*********************100%***********************]  1 of 1 completed
C:\Users\suhit\AppData\Local\Temp\ipykernel_13732\1092102379.py:17: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat({


In [25]:
print(market_ret_df.columns)

Index(['PLY.AX', 'LAU.AX', 'COS.AX', 'ANG.AX', 'VVA.AX', 'WTC.AX', 'AUB.AX',
       'TLX.AX', 'DUG.AX'],
      dtype='str')


In [26]:
betas = pd.DataFrame()
r_sq = {}

for stock in tickers:
    
    y = df['return'][stock]
    
    X = pd.concat([
        df['vol'][stock],
        df['momentum'][stock],
        df['liquidity'][stock],
        df[('market performance', stock)]
    ], axis=1)
    
    X.columns = ['vol', 'momentum', 'liquidity', 'market performance']
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={'maxlags':5})
    
    betas[stock] = model.params
    r_sq[stock] = model.rsquared

betas = betas.T
r_sq = pd.Series(r_sq)

print(betas)
print(f"\n{r_sq}")

           const       vol  momentum  liquidity  market performance
PLY.AX  0.195305 -2.206953  0.124107  -0.005952            1.732079
LAU.AX  0.035832  3.846127 -0.035839  -0.007478            0.811685
COS.AX  0.364546 -3.469920 -0.082753  -0.020272            0.520601
ANG.AX  0.342350 -1.629450 -0.018302  -0.016949            1.025064
VVA.AX -0.462529  2.326441 -0.030922   0.027788            0.586132
WTC.AX  1.726166  0.889606 -0.046087  -0.106222            1.777073
AUB.AX  0.777422 -3.125417 -0.166230  -0.046178            0.599643
TLX.AX  1.539497 -0.647224 -0.064958  -0.087921            1.009187
DUG.AX -0.236050  7.160764  0.215557  -0.000350            0.382288

PLY.AX    0.186035
LAU.AX    0.149431
COS.AX    0.345470
ANG.AX    0.147747
VVA.AX    0.134070
WTC.AX    0.325181
AUB.AX    0.318493
TLX.AX    0.119769
DUG.AX    0.244896
dtype: float64


In [27]:
factor_returns = {}

for month in df.index:
    y = df.loc[month, 'return']   
    
    X = pd.concat([
        df.loc[month, 'vol'],
        df.loc[month, 'momentum'],
        df.loc[month, 'liquidity']
    ], axis=1)
    
    X.columns = ['vol', 'momentum', 'liquidity']
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    factor_returns[month] = model.params

factor_returns = pd.DataFrame(factor_returns).T

print(factor_returns)

               const        vol  momentum  liquidity
2021-08-31 -1.003606   6.119365  0.055688   0.054348
2021-09-30 -0.174369   8.282447 -0.053514  -0.000927
2021-10-31 -0.689897  16.014407 -0.118223   0.019539
2021-11-30  0.610557  -1.024021  0.256528  -0.040651
2021-12-31 -0.552643   1.369756  0.113397   0.037177
2022-01-31  0.022353   3.712723 -0.139821  -0.014757
2022-02-28  0.572042   2.373125 -0.115984  -0.044717
2022-03-31 -0.096304  -5.663858  0.156376   0.018998
2022-04-30  0.317203   1.603198 -0.137872  -0.026443
2022-05-31  0.078029  -6.750569 -0.665810   0.001750
2022-06-30 -0.180639   3.027069  1.310025   0.005556
2022-07-31 -0.711123   3.414014  0.110873   0.051913
2022-08-31 -1.202433  -0.361820 -0.598671   0.083339
2022-09-30  0.553338  -1.914006  0.276168  -0.042407
2022-10-31 -1.654117   4.657558 -0.001445   0.101825
2022-11-30  0.314423  -1.179617 -0.303234  -0.012670
2022-12-31 -0.122376   7.175571  0.415289  -0.005007
2023-01-31  0.524016  -0.889194 -0.169081  -0.

In [47]:
factor_cols = ['vol', 'momentum', 'liquidity', 'market performance']
df_std = df.copy()

for factor in factor_cols:
    raw     = df[factor]                        # (n_months, n_stocks)
    cs_mean = raw.mean(axis=1)                  # mean across stocks per month
    cs_std  = raw.std(axis=1, ddof=1)           # std  across stocks per month

    df_std[factor] = raw.sub(cs_mean, axis=0).div(cs_std, axis=0)

In [46]:
betas_z = pd.DataFrame()
r_sq_z  = {}
cs_residuals = {}

for stock in tickers:
    y = df_std['return'][stock]

    X = pd.concat([
        df_std['vol'][stock],
        df_std['momentum'][stock],
        df_std['liquidity'][stock],
    ], axis=1)

    X.columns = ['vol', 'momentum', 'liquidity']
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

    betas_z[stock]       = model.params
    r_sq_z[stock]        = model.rsquared
    cs_residuals[stock]  = model.resid

betas_z      = betas_z.T
r_sq_z       = pd.Series(r_sq_z)
cs_residuals = pd.DataFrame(cs_residuals, index=df_std.index)

print(betas_z)
print(f"\n{r_sq_z}")

           const       vol  momentum  liquidity
PLY.AX  0.001801  0.003608  0.037778  -0.016585
LAU.AX  0.015491  0.024076  0.011310   0.016285
COS.AX -0.066218 -0.042542 -0.036709  -0.028237
ANG.AX  0.023899 -0.025055 -0.008559  -0.014985
VVA.AX  0.005041  0.056133 -0.029624  -0.007869
WTC.AX  0.070864  0.001392 -0.024145  -0.103720
AUB.AX -0.054082 -0.046693 -0.034564  -0.076567
TLX.AX  0.064552  0.011021 -0.000294  -0.046597
DUG.AX -0.020697  0.087517  0.037200  -0.002739

PLY.AX    0.064746
LAU.AX    0.048996
COS.AX    0.155714
ANG.AX    0.025183
VVA.AX    0.146358
WTC.AX    0.039122
AUB.AX    0.203376
TLX.AX    0.007511
DUG.AX    0.161172
dtype: float64


In [49]:
factor_returns_z = {}

for month in df_std.index:
    y = df_std.loc[month, 'return']

    X = pd.concat([
        df_std.loc[month, 'vol'],
        df_std.loc[month, 'momentum'],
        df_std.loc[month, 'liquidity'],
    ], axis=1)

    X.columns = ['vol', 'momentum', 'liquidity']
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    factor_returns_z[month] = model.params

factor_returns_z = pd.DataFrame(factor_returns_z).T

print(factor_returns_z)

               const       vol  momentum  liquidity
2021-08-31  0.079895  0.079107  0.012581   0.073083
2021-09-30  0.053611  0.096315 -0.015841  -0.001004
2021-10-31  0.026452  0.251754 -0.035782   0.025873
2021-11-30 -0.004953 -0.014718  0.092814  -0.050625
2021-12-31  0.065781  0.018899  0.045277   0.056199
2022-01-31 -0.059759  0.071492 -0.058175  -0.023600
2022-02-28 -0.040017  0.044859 -0.020193  -0.065172
2022-03-31  0.012110 -0.060690  0.029116   0.030191
2022-04-30 -0.017007  0.023433 -0.034683  -0.044739
2022-05-31 -0.101068 -0.081319 -0.120123   0.002290
2022-06-30 -0.107935  0.036388  0.068181   0.007383
2022-07-31  0.139211  0.044991  0.021929   0.086345
2022-08-31  0.121669 -0.004669 -0.164557   0.117562
2022-09-30 -0.092552 -0.017682  0.060741  -0.084705
2022-10-31  0.045989  0.045002 -0.000213   0.132430
2022-11-30  0.066926 -0.013198 -0.057578  -0.019314
2022-12-31  0.009938  0.076714  0.046903  -0.006231
2023-01-31  0.052248 -0.011268 -0.029607  -0.022085
2023-02-28  

In [52]:
covmatrix_z = factor_returns_z.drop(columns=['const']).cov()

print(covmatrix_z)

                vol  momentum  liquidity
vol        0.006249  0.000667   0.000479
momentum   0.000667  0.002655  -0.000259
liquidity  0.000479 -0.000259   0.002603


In [53]:
idio_var = cs_residuals.var(ddof=1)
D = np.diag(idio_var.values)

print(idio_var)

PLY.AX    0.040338
LAU.AX    0.011035
COS.AX    0.010916
ANG.AX    0.018310
VVA.AX    0.009226
WTC.AX    0.021615
AUB.AX    0.005200
TLX.AX    0.026809
DUG.AX    0.023333
dtype: float64


In [54]:
B = betas_z.drop(columns=['const'])[covmatrix_z.columns].values

Sigma = B @ covmatrix_z.values @ B.T + D

Sigma_df = pd.DataFrame(Sigma, index=tickers, columns=tickers)
print(Sigma_df)

              PLY.AX        LAU.AX        COS.AX        ANG.AX        VVA.AX  \
PLY.AX  4.034319e-02  1.334644e-06 -4.175036e-06 -1.145821e-06 -5.318417e-07   
LAU.AX  1.334644e-06  1.104048e-02 -1.003164e-05 -5.277157e-06  7.665487e-06   
COS.AX -4.175036e-06 -1.003164e-05  1.093548e-02  9.893256e-06 -1.288269e-05   
ANG.AX -1.145821e-06 -5.277157e-06  9.893256e-06  1.831575e-02 -8.075255e-06   
VVA.AX -5.318417e-07  7.665487e-06 -1.288269e-05 -8.075255e-06  9.245261e-03   
WTC.AX  2.783024e-06 -6.070277e-06  1.119232e-05  5.684187e-06 -6.001269e-08   
AUB.AX -1.636246e-06 -1.309492e-05  2.474863e-05  1.287877e-05 -1.500556e-05   
TLX.AX  2.793093e-06 -5.614654e-07  6.182343e-07  4.166202e-07  2.961787e-06   
DUG.AX  7.604076e-06  1.592987e-05 -3.077462e-05 -1.602117e-05  2.714526e-05   

              WTC.AX    AUB.AX        TLX.AX    DUG.AX  
PLY.AX  2.783024e-06 -0.000002  2.793093e-06  0.000008  
LAU.AX -6.070277e-06 -0.000013 -5.614654e-07  0.000016  
COS.AX  1.119232e-05  0.0000